In [26]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langchain.tools import tool
import requests
import os 
from dotenv import load_dotenv
from pprint import pprint
load_dotenv()

True

In [27]:
# Initialize Google API key and model

google_api_key = os.getenv("GEMINI_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash", api_key=google_api_key)


In [28]:
# Initialize Tavily search tool
tavily_api_key = os.getenv("TAVILY_API_KEY")
skill_demand_tool = TavilySearch(
    max_results=5,
    search_depth="advanced",
    tavily_api_key=tavily_api_key,
)


In [33]:
# Set up RapidAPI key and search jobs function
RAPIDAPI_KEY= os.getenv("RAPIDAPI_KEY")

@tool
def search_jobs(skill: str, location: str) -> list:
    """Search for jobs requiring a specific skill using JSearch API from RapidAPI."""
    print(f"\nCalling search_jobs tool")
    print(f"Searching jobs for: {skill} in {location}")

    url = "https://jsearch.p.rapidapi.com/search"
    headers = {
        "x-rapidapi-key": RAPIDAPI_KEY,
        "x-rapidapi-host": "jsearch.p.rapidapi.com"
    }
    querystring = {
        "query": f"{skill} in {location}",
        "page": "1",
        "country": "in",
        "employment_types": "INTERN,FULLTIME",
        "job_requirements": "no_experience,under_3_years_experience"
    }

    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()
    jobs = data.get("data", [])
    pprint(f"Found {len(jobs)} jobs")

# Format and return job results
    result = []
    for job in jobs:
        result.append({
            "title": job.get("job_title"),
            "company": job.get("employer_name"),
            "location": job.get("job_city"),
            "apply_link": job.get("job_apply_link")
        })
    return result







In [36]:
# Define system prompt for the agent
system_prompt = """You are a Skill-to-Career Mapping assistant that helps students understand skill demand and find matching job opportunities.

You have access to these tools:
- skill_demand_tool: Search for industry demand, salary insights, and career trends
- search_jobs: Find actual job listings requiring specific skills

Help the student by researching the skill they ask about and finding relevant opportunities.

Present results in a clean, readable format with clear sections and proper spacing. Include all job details with apply links. Don't use markdown format."""



In [37]:
# Create and invoke LangChain agent
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[skill_demand_tool, search_jobs],
    system_prompt=system_prompt,
    debug=True
)


In [40]:
user_query = "What's the demand for generative ai in the industry and show me related job openings in India"

response = agent.invoke({
    "messages": [{"role": "user", "content": user_query}]
})
print(response["messages"][-1].content[0]["text"])

[values] {'messages': [HumanMessage(content="What's the demand for generative ai in the industry and show me related job openings in India", additional_kwargs={}, response_metadata={}, id='5fa422fd-2855-4182-b72a-80fc55537625')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'search_jobs', 'arguments': '{"location": "India", "skill": "generative ai"}'}, '__gemini_function_call_thought_signatures__': {'cdf25255-b358-43fa-acc7-5522dac9ff90': 'CvQHAQw51sc2cSPL8iFUwbgRMoDqvs/ZzjEwYJmU+fOIG+eq0YeQds6Xt4r58V8VTrt09Ww/+uMxlFzMP0OuA3NEIUVCilxmPZj6YSDCVew3U1MchrFLysyuD92PhicbA2ogN0TJpQ+phB7dyQaGGlNk+PLW11KYgweWMVVXvRDO7KgEbBGw9mwoNEGC1y/QJMffLKlhQDjbAcJTznNvrwxyx/ekUrr1eAoY9Cc3Y20bkZGul5U5AHDFqXMrJP7q+GMPMbHRhlyrCVchCgkc6Un2ukRqqp5lJqX8yKg2o3eM6tIqTWugIeTRAQDE2ZXuXmlnkW2F1gH+s81Qfg2mYPJi7hpy3GGUhgatddSJ4uq+HqhngNjqp+q5VEkvB9/y8kMH/4GB9TVpE+R9R7lfs/zf5XKDVhMms+n6oR2c0HxTg9dLySPhaxuTZoHS8GSD7EjOiYs5nWBURmqocoKb5jL1Z8aWSQK3ogInhi9ih083fKEZktvy